# Step 3~5 정제 + 임베딩 청킹 (Colab · A100)

**용어집(3) → 섹션 정제(4, Solar-10.7B) → 임베딩 청킹+태깅(5, KURE)**. GPU 필요(Colab Pro A100).

입력 `merged.jsonl`(전처리 산출물 — 로컬 `scripts.run_preprocess` 또는 `00_preprocess_colab.ipynb`)을 Drive에 둔다.
- 정제(④): 섹션 1건씩 **체크포인트** → 끊겨도 재실행 시 재개.
- 청킹(⑤): KURE 임베딩으로 토픽 분할 + 평가항목 `eval_tags` 태깅(LLM 호출 0, 빠름).

### 사전 준비 (Drive `MyDrive/lecture-analyzer/`)
- 로컬 `src/` 폴더 업로드 (코드)
- `merged.jsonl` 업로드 (데이터)

> ⚠️ 보안: 데이터·산출물은 **개인 Drive에만**. git/공개 업로드 금지.

## 0. Drive 마운트 + 런타임 확인 (A100인지 확인)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 코드 연결 + 의존성 설치
**src 업로드 방식**(토큰 불필요): 로컬 `src/` 폴더를 `MyDrive/lecture-analyzer/`에 올려두면 import 가능.

In [ ]:
import sys
DRIVE = '/content/drive/MyDrive/lecture-analyzer'
sys.path.insert(0, DRIVE)        # 업로드한 src/ 를 import 경로에 추가
import src.config                 # 임포트 되면 코드 연결 성공
# 모델 추론 + 임베딩 의존성 (torch는 Colab 기본본 사용)
!pip install -q transformers==4.46.3 accelerate==1.1.1 sentencepiece==0.2.0 sentence-transformers==3.3.1
print('코드 연결 OK')

## 2. 경로 설정 (Drive 작업 폴더)
로컬에서 만든 `merged.jsonl`을 아래 폴더에 업로드해 둘 것.

In [ ]:
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/lecture-analyzer'); DRIVE.mkdir(parents=True, exist_ok=True)
MERGED   = DRIVE / 'merged.jsonl'          # ← 입력 (업로드 필요)
GLOSSARY = DRIVE / 'glossary.json'         # 확정 용어집(없으면 SEED 사용)
GLOSS_CAND = DRIVE / 'glossary_candidates.json'
CLEAN    = DRIVE / 'clean.jsonl'           # 산출 (체크포인트)
CHUNKS   = DRIVE / 'chunks.jsonl'          # 산출 (체크포인트)
assert MERGED.exists(), f'merged.jsonl 을 {DRIVE} 에 업로드하세요'
print('입력 확인:', MERGED)

## 3. Solar-10.7B 로드 (A100 40GB · fp16, 수 분 소요)

In [ ]:
from src.refine.model import load_solar, make_generate_fn
model, tok = load_solar()
generate_fn = make_generate_fn(model, tok)
print('모델 로드 완료:', model.config._name_or_path)

## 4. 섹션화 (블록 → 큰 섹션)

In [ ]:
from src.refine.sectionize import load_merged, make_sections
sections = make_sections(load_merged(MERGED))
print('섹션 수:', len(sections), '| 평균 글자:', sum(s['n_chars'] for s in sections)//len(sections))

## 5. (선택) 용어집 후보 추출 → 사람 검수
후보를 뽑아 `glossary_candidates.json`에 저장. 사람이 검수해 `glossary.json`(확정본)으로 옮긴다.
확정본이 없으면 다음 셀에서 SEED 용어집을 사용한다.

In [ ]:
from src.refine.glossary import build_candidates
cands = build_candidates(sections, generate_fn, GLOSS_CAND, sample_every=5)  # 5섹션마다 1회
print('correction 후보:', len(cands['corrections']), '| term 후보:', len(cands['terms']))
cands['corrections'][:10]

## 6. 정제 (Step 4) — 체크포인트. 끊기면 이 셀만 다시 실행하면 이어서 재개

In [ ]:
from src.refine.glossary import load_glossary
from src.refine.refine import run_refine
glossary = load_glossary(GLOSSARY)  # 확정본 없으면 SEED
stats_refine = run_refine(sections, glossary, generate_fn, CLEAN)
print(stats_refine)

## 7. 임베딩 청킹 + 평가항목 태깅 (Step 5 · KURE)
`clean.jsonl` → **KURE 임베딩**으로 토픽 분할 + `eval_tags` 다중 라벨 태깅.
임베딩 1회를 분할·태깅에 공유(추가 LLM 호출 0). 몇 분이면 끝 → 체크포인트 불필요(재실행 시 새로 씀).

In [ ]:
# KURE 임베딩 로드(가벼움) + 임베딩 기반 청킹 + 평가항목 태깅
from src.refine.embedding import load_embedder, make_embed_fn
from src.refine.chunk_embed import run_chunk_embed
embed_fn = make_embed_fn(load_embedder())          # nlpai-lab/KURE-v1
stats_chunk = run_chunk_embed(CLEAN, embed_fn, CHUNKS)
print(stats_chunk)

In [ ]:
# 결과 확인 — 항목별 태깅 커버리지 (0 = 부정 증거 후보) + 샘플
import json
from src.refine.tagging import coverage
chunks = [json.loads(l) for l in CHUNKS.open(encoding='utf-8')]
print('청크 수:', len(chunks))
print('항목별 태깅:', coverage(chunks))
print('샘플 eval_tags:', chunks[0]['eval_tags'][:5])

## 8. manifest 기록 (재현성)

In [ ]:
from src import config
from src.manifest import write_manifest
write_manifest(DRIVE / 'manifest_refine.json', step='refine+chunk(step3-5)',
    params={'model_id': config.MODEL_ID, 'embed_model': config.EMBED_MODEL_ID,
            'section_max_chars': config.SECTION_MAX_CHARS,
            'seg_depth_c': config.SEG_DEPTH_C, 'tag_sim_threshold': config.TAG_SIM_THRESHOLD,
            'seed': config.SEED},
    stats={'refine': stats_refine, 'chunk': stats_chunk}, inputs=[MERGED])
print('manifest 저장 완료')